Milvus CDC 实操

这是一个Milvus CDC 主备搭建实操教程，在本教程里基本前提以及内容简介如下

- Kubernetes 集群已可用；
- 使用 Milvus Operator 部署 Milvus CDC；
- Source/Primary 集群与 Target/Standby 集群部署在同一个 K8s 集群、同一个 namespace 中；
- Milvus 版本使用 v2.6.6+；
- Milvus Operator 版本要求 v1.3.4+；
- CDC 当前只支持单副本；
- 仅适用于全新集群；如果已有数据，需要先用 Milvus Backup 迁移历史数据，再开启 CDC。



1. 通过Milvus operator部署 Milvus CDC

```
#!/usr/bin/env bash
set -euo pipefail

OPERATOR_NS="milvus-operator"

echo "[1/5] Create namespace ${OPERATOR_NS}"
kubectl create namespace ${OPERATOR_NS} --dry-run=client -o yaml | kubectl apply -f -

echo "[2/5] Add Milvus Operator helm repo"
helm repo add zilliztech-milvus-operator https://zilliztech.github.io/milvus-operator/ || true

echo "[3/5] Update helm repo"
helm repo update zilliztech-milvus-operator

echo "[4/5] Install or upgrade Milvus Operator"
helm upgrade --install milvus-operator \
  zilliztech-milvus-operator/milvus-operator \
  -n ${OPERATOR_NS} \
  --wait

echo "[5/5] Check operator status"
kubectl rollout status -n ${OPERATOR_NS} deploy/milvus-operator --timeout=300s
kubectl get pods -n ${OPERATOR_NS}

echo "Milvus Operator installed/upgraded successfully."
```

执行上述脚本


```
❯ ./00-install-operator.sh
[1/5] Create namespace milvus-operator
namespace/milvus-operator created

[2/5] Add Milvus Operator helm repo
"zilliztech-milvus-operator" has been added to your repositories

[3/5] Update helm repo
Hang tight while we grab the latest from your chart repositories...
...Successfully got an update from the "zilliztech-milvus-operator" chart repository
Update Complete. ⎈Happy Helming!⎈

[4/5] Install or upgrade Milvus Operator
Release "milvus-operator" does not exist. Installing it now.
NAME: milvus-operator
LAST DEPLOYED: Tue Jun  2 00:14:16 2026
NAMESPACE: milvus-operator
STATUS: deployed
REVISION: 1
TEST SUITE: None
NOTES:
Milvus Operator Is Starting, use `kubectl get -n milvus-operator deploy/milvus-operator` to check if its successfully installed
Full Installation doc can be found in https://github.com/zilliztech/milvus-operator/blob/main/docs/installation/installation.md
Quick start with `kubectl apply -f https://raw.githubusercontent.com/zilliztech/milvus-operator/main/config/samples/milvus_minimum.yaml`
More samples can be found in https://github.com/zilliztech/milvus-operator/tree/main/config/samples
CRD Documentation can be found in https://github.com/zilliztech/milvus-operator/tree/main/docs/CRD
Administration Documentation can be found in https://github.com/zilliztech/milvus-operator/tree/main/docs/administration

[5/5] Check operator status
deployment "milvus-operator" successfully rolled out
NAME                               READY   STATUS    RESTARTS   AGE
milvus-operator-77445f9484-j4kwd   1/1     Running   0          22s
Milvus Operator installed/upgraded successfully.
```


2. 部署 Source 主集群

Source 集群是唯一写入点，负责业务写入、DDL、DML，并通过 CDC 将 WAL 变更同步到 Target。

```
apiVersion: milvus.io/v1beta1
kind: Milvus
metadata:
  name: source-cluster
  namespace: milvus
  labels:
    app: milvus
    role: source
spec:
  mode: cluster

  components:
    # CDC 从 Milvus 2.6.6 开始支持
    image: milvusdb/milvus:v2.6.6

    # Source 集群需要启用 CDC 组件
    cdc:
      # 当前 Milvus CDC 仅支持单副本
      replicas: 1

  dependencies:
    # Milvus 2.6 CDC 示例使用 woodpecker
    msgStreamType: woodpecker
```

3. 部署 Target 备集群

Target 集群作为热备副本，正常情况下只承担读请求，不建议直接写入。

```
apiVersion: milvus.io/v1beta1
kind: Milvus
metadata:
  name: target-cluster
  namespace: milvus
  labels:
    app: milvus
    role: target
spec:
  mode: cluster

  components:
    image: milvusdb/milvus:v2.6.6

  dependencies:
    msgStreamType: woodpecker
```

注意：Target 集群不需要配置 cdc 组件。CDC Node 部署在 Source 侧，负责消费 Source WAL 并转发到 Target。


4. 应用 Source/Target 集群配置

```
#!/usr/bin/env bash
set -euo pipefail

MILVUS_NS="milvus"

echo "[1/3] Create namespace ${MILVUS_NS}"
kubectl create namespace ${MILVUS_NS} --dry-run=client -o yaml | kubectl apply -f -

echo "[2/3] Apply source cluster"
kubectl apply -f 01-source-cluster.yaml

echo "[3/3] Apply target cluster"
kubectl apply -f 02-target-cluster.yaml

echo "Milvus source and target cluster manifests applied."
echo "Use ./04-check-clusters.sh to check status."
```

执行：

```
========== Milvus CR ==========
NAME             MODE      STATUS    UPDATED   AGE
source-cluster   cluster   Healthy   True      10m
target-cluster   cluster   Healthy   True      10m

========== Pods ==========
NAME                                                   READY   STATUS    RESTARTS   AGE
source-cluster-etcd-0                                  1/1     Running   0          10m
source-cluster-etcd-1                                  1/1     Running   0          10m
source-cluster-etcd-2                                  1/1     Running   0          10m
source-cluster-milvus-cdc-6776656874-zcjjf             1/1     Running   0          4m21s
source-cluster-milvus-datanode-688dd89544-pdbt5        1/1     Running   0          4m21s
source-cluster-milvus-mixcoord-67887b88db-52c4q        1/1     Running   0          4m21s
source-cluster-milvus-proxy-7dd775d9b4-zbnhs           1/1     Running   0          4m21s
source-cluster-milvus-querynode-0-5bbcf78595-d2pww     1/1     Running   0          4m20s
source-cluster-milvus-streamingnode-6c4c775d6-znfjc    1/1     Running   0          4m21s
source-cluster-minio-0                                 1/1     Running   0          10m
source-cluster-minio-1                                 1/1     Running   0          10m
source-cluster-minio-2                                 1/1     Running   0          10m
source-cluster-minio-3                                 1/1     Running   0          10m
target-cluster-etcd-0                                  1/1     Running   0          10m
target-cluster-etcd-1                                  1/1     Running   0          10m
target-cluster-etcd-2                                  1/1     Running   0          10m
target-cluster-milvus-datanode-76845cb85b-gnwtt        1/1     Running   0          8m52s
target-cluster-milvus-mixcoord-f4c5d5cbb-q8b6t         1/1     Running   0          8m52s
target-cluster-milvus-proxy-b6b5f7464-fqzkp            1/1     Running   0          8m52s
target-cluster-milvus-querynode-0-8954dd78c-2z4zt      1/1     Running   0          8m51s
target-cluster-milvus-streamingnode-7d55c58646-btvbc   1/1     Running   0          8m52s
target-cluster-minio-0                                 1/1     Running   0          10m
target-cluster-minio-1                                 1/1     Running   0          10m
target-cluster-minio-2                                 1/1     Running   0          10m
target-cluster-minio-3                                 1/1     Running   0          10m

========== Services ==========
NAME                           TYPE        CLUSTER-IP       EXTERNAL-IP   PORT(S)              AGE
source-cluster-etcd            ClusterIP   10.100.213.117   <none>        2379/TCP,2380/TCP    10m
source-cluster-etcd-headless   ClusterIP   None             <none>        2379/TCP,2380/TCP    10m
source-cluster-milvus          ClusterIP   10.100.180.221   <none>        19530/TCP,9091/TCP   4m22s
source-cluster-minio           ClusterIP   10.100.200.247   <none>        9000/TCP             10m
source-cluster-minio-svc       ClusterIP   None             <none>        9000/TCP             10m
target-cluster-etcd            ClusterIP   10.100.26.208    <none>        2379/TCP,2380/TCP    10m
target-cluster-etcd-headless   ClusterIP   None             <none>        2379/TCP,2380/TCP    10m
target-cluster-milvus          ClusterIP   10.100.111.61    <none>        19530/TCP,9091/TCP   8m53s
target-cluster-minio           ClusterIP   10.100.201.212   <none>        9000/TCP             10m
target-cluster-minio-svc       ClusterIP   None             <none>        9000/TCP             10m

========== Source CDC Pod ==========
source-cluster-milvus-cdc-6776656874-zcjjf             1/1     Running   0          4m24s
```



5. 创建 CDC 复制关系并验证同步

通过一个 K8s Job来验证CDC完成了两个milvus集群数据同步

- 连接 Source 和 Target；
- 调用 update_replicate_configuration 配置 CDC 拓扑；
- 在 Source 创建 Collection、插入数据、查询；
- 轮询 Target，确认 Collection、数据、Search 结果已同步。

```
apiVersion: v1
kind: ConfigMap
metadata:
  name: milvus-cdc-config-verify-script
  namespace: milvus
data:
  cdc_config_verify.py: |
    import os
    import time
    import math
    import traceback
    from pymilvus import MilvusClient, DataType

    SOURCE_CLUSTER_ID = os.getenv("SOURCE_CLUSTER_ID", "source-cluster")
    TARGET_CLUSTER_ID = os.getenv("TARGET_CLUSTER_ID", "target-cluster")

    SOURCE_CLUSTER_ADDR = os.getenv(
        "SOURCE_CLUSTER_ADDR",
        "http://source-cluster-milvus.milvus.svc.cluster.local:19530",
    )
    TARGET_CLUSTER_ADDR = os.getenv(
        "TARGET_CLUSTER_ADDR",
        "http://target-cluster-milvus.milvus.svc.cluster.local:19530",
    )

    SOURCE_CLUSTER_TOKEN = os.getenv("SOURCE_CLUSTER_TOKEN", "root:Milvus")
    TARGET_CLUSTER_TOKEN = os.getenv("TARGET_CLUSTER_TOKEN", "root:Milvus")

    # 默认 Milvus dml pchannel 数为 16。
    # 如果你修改过 Milvus 内部 channel 配置，这里必须改成实际值。
    PCHANNEL_NUM = int(os.getenv("PCHANNEL_NUM", "16"))

    VECTOR_DIM = int(os.getenv("VECTOR_DIM", "4"))
    ROW_COUNT = int(os.getenv("ROW_COUNT", "20"))

    VERIFY_TIMEOUT_SECONDS = int(os.getenv("VERIFY_TIMEOUT_SECONDS", "180"))
    VERIFY_INTERVAL_SECONDS = int(os.getenv("VERIFY_INTERVAL_SECONDS", "3"))

    RUN_ID = str(int(time.time()))
    COLLECTION_NAME = os.getenv("COLLECTION_NAME", f"cdc_verify_{RUN_ID}")

    def build_pchannels(cluster_id: str, num: int):
        return [f"{cluster_id}-rootcoord-dml_{i}" for i in range(num)]

    def build_replicate_config():
        source_pchannels = build_pchannels(SOURCE_CLUSTER_ID, PCHANNEL_NUM)
        target_pchannels = build_pchannels(TARGET_CLUSTER_ID, PCHANNEL_NUM)

        config = {
            "clusters": [
                {
                    "cluster_id": SOURCE_CLUSTER_ID,
                    "connection_param": {
                        "uri": SOURCE_CLUSTER_ADDR,
                        "token": SOURCE_CLUSTER_TOKEN,
                    },
                    "pchannels": source_pchannels,
                },
                {
                    "cluster_id": TARGET_CLUSTER_ID,
                    "connection_param": {
                        "uri": TARGET_CLUSTER_ADDR,
                        "token": TARGET_CLUSTER_TOKEN,
                    },
                    "pchannels": target_pchannels,
                },
            ],
            "cross_cluster_topology": [
                {
                    "source_cluster_id": SOURCE_CLUSTER_ID,
                    "target_cluster_id": TARGET_CLUSTER_ID,
                }
            ],
        }
        return config

    def normalize(vec):
        s = math.sqrt(sum(x * x for x in vec))
        return [x / s for x in vec]

    def make_vector(i: int):
        # 构造稳定、可搜索的向量
        base = [0.0] * VECTOR_DIM
        base[i % VECTOR_DIM] = 1.0
        for j in range(VECTOR_DIM):
            base[j] += i * 0.001
        return normalize(base)

    def create_collection(client: MilvusClient, collection_name: str):
        print(f"[Source] Creating collection: {collection_name}")

        schema = client.create_schema(
            auto_id=False,
            enable_dynamic_field=False,
        )

        schema.add_field(
            field_name="id",
            datatype=DataType.INT64,
            is_primary=True,
        )
        schema.add_field(
            field_name="vector",
            datatype=DataType.FLOAT_VECTOR,
            dim=VECTOR_DIM,
        )
        schema.add_field(
            field_name="tag",
            datatype=DataType.VARCHAR,
            max_length=128,
        )

        index_params = client.prepare_index_params()
        index_params.add_index(
            field_name="vector",
            index_type="AUTOINDEX",
            metric_type="COSINE",
        )

        client.create_collection(
            collection_name=collection_name,
            schema=schema,
            index_params=index_params,
        )

    def insert_data(client: MilvusClient, collection_name: str):
        print(f"[Source] Inserting {ROW_COUNT} rows")

        rows = []
        for i in range(ROW_COUNT):
            rows.append(
                {
                    "id": i,
                    "vector": make_vector(i),
                    "tag": f"cdc-{RUN_ID}-{i}",
                }
            )

        result = client.insert(
            collection_name=collection_name,
            data=rows,
        )
        print(f"[Source] Insert result: {result}")

        print("[Source] Flushing")
        client.flush(collection_name=collection_name)

    def load_collection(client: MilvusClient, collection_name: str, cluster_name: str):
        print(f"[{cluster_name}] Loading collection: {collection_name}")
        client.load_collection(collection_name=collection_name)

    def query_all(client: MilvusClient, collection_name: str, cluster_name: str):
        print(f"[{cluster_name}] Query all rows")
        rows = client.query(
            collection_name=collection_name,
            filter="id >= 0",
            output_fields=["id", "tag"],
            limit=ROW_COUNT,
        )
        rows = sorted(rows, key=lambda x: x["id"])
        print(f"[{cluster_name}] Query result count: {len(rows)}")
        print(f"[{cluster_name}] Query sample: {rows[:5]}")
        return rows

    def search_topk(client: MilvusClient, collection_name: str, cluster_name: str):
        print(f"[{cluster_name}] Search topK")
        result = client.search(
            collection_name=collection_name,
            data=[make_vector(0)],
            anns_field="vector",
            search_params={
                "metric_type": "COSINE",
                "params": {},
            },
            limit=5,
            output_fields=["id", "tag"],
        )
        print(f"[{cluster_name}] Search result: {result}")
        return result

    def wait_target_synced(target_client: MilvusClient, collection_name: str):
        print(f"[Target] Waiting CDC sync, timeout={VERIFY_TIMEOUT_SECONDS}s")

        deadline = time.time() + VERIFY_TIMEOUT_SECONDS
        last_error = None

        while time.time() < deadline:
            try:
                if not target_client.has_collection(collection_name):
                    print(f"[Target] Collection not found yet: {collection_name}")
                    time.sleep(VERIFY_INTERVAL_SECONDS)
                    continue

                # 注意：Target 是 CDC 备集群（secondary），禁止任何 DDL/DCL。
                # 不能调用 load_collection（会报 "cluster is not primary"）。
                # load 状态由 Source 通过 CDC 复制过来，这里只做 query 并重试。
                rows = target_client.query(
                    collection_name=collection_name,
                    filter="id >= 0",
                    output_fields=["id", "tag"],
                    limit=ROW_COUNT,
                )

                print(f"[Target] Current synced rows: {len(rows)}/{ROW_COUNT}")

                if len(rows) >= ROW_COUNT:
                    rows = sorted(rows, key=lambda x: x["id"])
                    return rows

            except Exception as e:
                last_error = e
                print(f"[Target] Not ready yet: {repr(e)}")

            time.sleep(VERIFY_INTERVAL_SECONDS)

        raise TimeoutError(
            f"Target cluster did not sync expected rows within "
            f"{VERIFY_TIMEOUT_SECONDS}s. Last error: {repr(last_error)}"
        )

    def configure_cdc():
        config = build_replicate_config()

        print("========== CDC Replicate Configuration ==========")
        print(config)

        source_client = MilvusClient(
            uri=SOURCE_CLUSTER_ADDR,
            token=SOURCE_CLUSTER_TOKEN,
        )
        target_client = MilvusClient(
            uri=TARGET_CLUSTER_ADDR,
            token=TARGET_CLUSTER_TOKEN,
        )

        try:
            print("[Source] update_replicate_configuration")
            source_client.update_replicate_configuration(**config)

            print("[Target] update_replicate_configuration")
            target_client.update_replicate_configuration(**config)

            print("CDC replicate configuration updated successfully.")
        finally:
            source_client.close()
            target_client.close()

    def verify_cdc():
        source_client = MilvusClient(
            uri=SOURCE_CLUSTER_ADDR,
            token=SOURCE_CLUSTER_TOKEN,
        )
        target_client = MilvusClient(
            uri=TARGET_CLUSTER_ADDR,
            token=TARGET_CLUSTER_TOKEN,
        )

        try:
            print("========== Source Create Collection ==========")
            create_collection(source_client, COLLECTION_NAME)

            print("========== Source Insert Data ==========")
            insert_data(source_client, COLLECTION_NAME)

            print("========== Source Load / Query / Search ==========")
            load_collection(source_client, COLLECTION_NAME, "Source")
            source_rows = query_all(source_client, COLLECTION_NAME, "Source")
            source_search = search_topk(source_client, COLLECTION_NAME, "Source")

            if len(source_rows) != ROW_COUNT:
                raise RuntimeError(
                    f"Source row count mismatch: expected={ROW_COUNT}, actual={len(source_rows)}"
                )

            print("========== Target Wait CDC Sync ==========")
            target_rows = wait_target_synced(target_client, COLLECTION_NAME)

            print("========== Target Query / Search ==========")
            print(f"[Target] Query result count: {len(target_rows)}")
            print(f"[Target] Query sample: {target_rows[:5]}")
            target_search = search_topk(target_client, COLLECTION_NAME, "Target")

            source_ids = [r["id"] for r in source_rows]
            target_ids = [r["id"] for r in target_rows]

            if source_ids != target_ids:
                raise RuntimeError(
                    f"Source/Target ids mismatch. source={source_ids}, target={target_ids}"
                )

            print("========== CDC VERIFY PASSED ==========")
            print(f"Collection: {COLLECTION_NAME}")
            print(f"Source rows: {len(source_rows)}")
            print(f"Target rows: {len(target_rows)}")
            print(f"Source search: {source_search}")
            print(f"Target search: {target_search}")

        finally:
            source_client.close()
            target_client.close()

    def main():
        print("========== Milvus CDC Config And Verify ==========")
        print(f"SOURCE_CLUSTER_ID   = {SOURCE_CLUSTER_ID}")
        print(f"TARGET_CLUSTER_ID   = {TARGET_CLUSTER_ID}")
        print(f"SOURCE_CLUSTER_ADDR = {SOURCE_CLUSTER_ADDR}")
        print(f"TARGET_CLUSTER_ADDR = {TARGET_CLUSTER_ADDR}")
        print(f"PCHANNEL_NUM        = {PCHANNEL_NUM}")
        print(f"COLLECTION_NAME     = {COLLECTION_NAME}")

        configure_cdc()
        verify_cdc()

    if __name__ == "__main__":
        try:
            main()
        except Exception:
            traceback.print_exc()
            raise

---
apiVersion: batch/v1
kind: Job
metadata:
  name: milvus-cdc-config-verify
  namespace: milvus
spec:
  backoffLimit: 0
  template:
    metadata:
      labels:
        app: milvus-cdc-config-verify
    spec:
      restartPolicy: Never
      containers:
        - name: cdc-config-verify
          image: python:3.11-slim
          imagePullPolicy: IfNotPresent
          env:
            - name: SOURCE_CLUSTER_ID
              value: "source-cluster"
            - name: TARGET_CLUSTER_ID
              value: "target-cluster"

            # 注意：这里必须使用 CDC Pod 可以访问到的地址。
            # 同 namespace 下也可以写成：
            # http://source-cluster-milvus:19530
            # http://target-cluster-milvus:19530
            - name: SOURCE_CLUSTER_ADDR
              value: "http://source-cluster-milvus.milvus.svc.cluster.local:19530"
            - name: TARGET_CLUSTER_ADDR
              value: "http://target-cluster-milvus.milvus.svc.cluster.local:19530"

            - name: SOURCE_CLUSTER_TOKEN
              value: "root:Milvus"
            - name: TARGET_CLUSTER_TOKEN
              value: "root:Milvus"

            # 默认 16。如果你的 Milvus pchannel 数调整过，必须同步修改。
            - name: PCHANNEL_NUM
              value: "16"

            - name: VECTOR_DIM
              value: "4"
            - name: ROW_COUNT
              value: "20"
            - name: VERIFY_TIMEOUT_SECONDS
              value: "180"
            - name: VERIFY_INTERVAL_SECONDS
              value: "3"

          command:
            - /bin/bash
            - -c
            - |
              set -euo pipefail
              pip install --no-cache-dir "pymilvus>=2.6.6"
              python /scripts/cdc_config_verify.py

          volumeMounts:
            - name: script
              mountPath: /scripts

      volumes:
        - name: script
          configMap:
            name: milvus-cdc-config-verify-script
```

 执行 CDC 配置和验证

应用 Job：

kubectl apply -f cdc-config-verify.yaml

查看日志：

kubectl logs -f job/milvus-cdc-config-verify -n milvus


如果正常，会看到类似：

```
❯ kubectl logs -n milvus -l app=milvus-cdc-config-verify -f
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.6/44.6 kB 340.3 MB/s eta 0:00:00
Downloading urllib3-2.7.0-py3-none-any.whl (131 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.1/131.1 kB 398.8 MB/s eta 0:00:00
Downloading six-1.17.0-py2.py3-none-any.whl (11 kB)
Installing collected packages: urllib3, typing-extensions, six, python-dotenv, protobuf, orjson, numpy, idna, charset_normalizer, certifi, cachetools, requests, python-dateutil, grpcio, pandas, pymilvus
Successfully installed cachetools-7.1.4 certifi-2026.5.20 charset_normalizer-3.4.7 grpcio-1.81.0 idna-3.17 numpy-2.4.6 orjson-3.11.9 pandas-3.0.3 protobuf-7.35.0 pymilvus-3.0.0 python-dateutil-2.9.0.post0 python-dotenv-1.2.2 requests-2.34.2 six-1.17.0 typing-extensions-4.15.0 urllib3-2.7.0
WARNING: Running pip as the 'root' user can result in broken permissions and conflicting behaviour with the system package manager. It is recommended to use a virtual environment instead: https://pip.pypa.io/warnings/venv

[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
========== Milvus CDC Config And Verify ==========
SOURCE_CLUSTER_ID   = source-cluster
TARGET_CLUSTER_ID   = target-cluster
SOURCE_CLUSTER_ADDR = http://source-cluster-milvus.milvus.svc.cluster.local:19530
TARGET_CLUSTER_ADDR = http://target-cluster-milvus.milvus.svc.cluster.local:19530
PCHANNEL_NUM        = 16
COLLECTION_NAME     = cdc_verify_1780392387
========== CDC Replicate Configuration ==========
{'clusters': [{'cluster_id': 'source-cluster', 'connection_param': {'uri': 'http://source-cluster-milvus.milvus.svc.cluster.local:19530', 'token': 'root:Milvus'}, 'pchannels': ['source-cluster-rootcoord-dml_0', 'source-cluster-rootcoord-dml_1', 'source-cluster-rootcoord-dml_2', 'source-cluster-rootcoord-dml_3', 'source-cluster-rootcoord-dml_4', 'source-cluster-rootcoord-dml_5', 'source-cluster-rootcoord-dml_6', 'source-cluster-rootcoord-dml_7', 'source-cluster-rootcoord-dml_8', 'source-cluster-rootcoord-dml_9', 'source-cluster-rootcoord-dml_10', 'source-cluster-rootcoord-dml_11', 'source-cluster-rootcoord-dml_12', 'source-cluster-rootcoord-dml_13', 'source-cluster-rootcoord-dml_14', 'source-cluster-rootcoord-dml_15']}, {'cluster_id': 'target-cluster', 'connection_param': {'uri': 'http://target-cluster-milvus.milvus.svc.cluster.local:19530', 'token': 'root:Milvus'}, 'pchannels': ['target-cluster-rootcoord-dml_0', 'target-cluster-rootcoord-dml_1', 'target-cluster-rootcoord-dml_2', 'target-cluster-rootcoord-dml_3', 'target-cluster-rootcoord-dml_4', 'target-cluster-rootcoord-dml_5', 'target-cluster-rootcoord-dml_6', 'target-cluster-rootcoord-dml_7', 'target-cluster-rootcoord-dml_8', 'target-cluster-rootcoord-dml_9', 'target-cluster-rootcoord-dml_10', 'target-cluster-rootcoord-dml_11', 'target-cluster-rootcoord-dml_12', 'target-cluster-rootcoord-dml_13', 'target-cluster-rootcoord-dml_14', 'target-cluster-rootcoord-dml_15']}], 'cross_cluster_topology': [{'source_cluster_id': 'source-cluster', 'target_cluster_id': 'target-cluster'}]}
[Source] update_replicate_configuration
[Target] update_replicate_configuration
CDC replicate configuration updated successfully.
========== Source Create Collection ==========
[Source] Creating collection: cdc_verify_1780392387
========== Source Insert Data ==========
[Source] Inserting 20 rows
[Source] Insert result: {'insert_count': 20, 'ids': [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19]}
[Source] Flushing
========== Source Load / Query / Search ==========
[Source] Loading collection: cdc_verify_1780392387
[Source] Query all rows
[Source] Query result count: 20
[Source] Query sample: [{'id': 0, 'tag': 'cdc-1780392387-0'}, {'id': 1, 'tag': 'cdc-1780392387-1'}, {'id': 2, 'tag': 'cdc-1780392387-2'}, {'id': 3, 'tag': 'cdc-1780392387-3'}, {'id': 4, 'tag': 'cdc-1780392387-4'}]
[Source] Search topK
[Source] Search result: data: [[{'id': 0, 'distance': 1.0, 'entity': {'tag': 'cdc-1780392387-0', 'id': 0}}, {'id': 4, 'distance': 0.9999762177467346, 'entity': {'tag': 'cdc-1780392387-4', 'id': 4}}, {'id': 8, 'distance': 0.999905526638031, 'entity': {'tag': 'cdc-1780392387-8', 'id': 8}}, {'id': 12, 'distance': 0.9997891783714294, 'entity': {'tag': 'cdc-1780392387-12', 'id': 12}}, {'id': 16, 'distance': 0.9996282458305359, 'entity': {'tag': 'cdc-1780392387-16', 'id': 16}}]]
========== Target Wait CDC Sync ==========
[Target] Waiting CDC sync, timeout=180s
[Target] Current synced rows: 20/20
========== Target Query / Search ==========
[Target] Query result count: 20
[Target] Query sample: [{'id': 0, 'tag': 'cdc-1780392387-0'}, {'id': 1, 'tag': 'cdc-1780392387-1'}, {'id': 2, 'tag': 'cdc-1780392387-2'}, {'id': 3, 'tag': 'cdc-1780392387-3'}, {'id': 4, 'tag': 'cdc-1780392387-4'}]
[Target] Search topK
[Target] Search result: data: [[{'id': 0, 'distance': 1.0, 'entity': {'tag': 'cdc-1780392387-0', 'id': 0}}, {'id': 4, 'distance': 0.9999762177467346, 'entity': {'tag': 'cdc-1780392387-4', 'id': 4}}, {'id': 8, 'distance': 0.999905526638031, 'entity': {'tag': 'cdc-1780392387-8', 'id': 8}}, {'id': 12, 'distance': 0.9997891783714294, 'entity': {'tag': 'cdc-1780392387-12', 'id': 12}}, {'id': 16, 'distance': 0.9996282458305359, 'entity': {'tag': 'cdc-1780392387-16', 'id': 16}}]]
========== CDC VERIFY PASSED ==========
Collection: cdc_verify_1780392387
Source rows: 20
Target rows: 20
Source search: data: [[{'id': 0, 'distance': 1.0, 'entity': {'tag': 'cdc-1780392387-0', 'id': 0}}, {'id': 4, 'distance': 0.9999762177467346, 'entity': {'tag': 'cdc-1780392387-4', 'id': 4}}, {'id': 8, 'distance': 0.999905526638031, 'entity': {'tag': 'cdc-1780392387-8', 'id': 8}}, {'id': 12, 'distance': 0.9997891783714294, 'entity': {'tag': 'cdc-1780392387-12', 'id': 12}}, {'id': 16, 'distance': 0.9996282458305359, 'entity': {'tag': 'cdc-1780392387-16', 'id': 16}}]]
Target search: data: [[{'id': 0, 'distance': 1.0, 'entity': {'tag': 'cdc-1780392387-0', 'id': 0}}, {'id': 4, 'distance': 0.9999762177467346, 'entity': {'tag': 'cdc-1780392387-4', 'id': 4}}, {'id': 8, 'distance': 0.999905526638031, 'entity': {'tag': 'cdc-1780392387-8', 'id': 8}}, {'id': 12, 'distance': 0.9997891783714294, 'entity': {'tag': 'cdc-1780392387-12', 'id': 12}}, {'id': 16, 'distance': 0.9996282458305359, 'entity': {'tag': 'cdc-1780392387-16', 'id': 16}}]]
```

Milvus CDC完成了两个Milvus集群数据同步

详细代码参见 (./cdc_demo)


6. 关键注意事项

-  Source 是唯一写入点

生产中应保证：

- DDL 只写 Source；
- DML 只写 Source；
- DCL 尽量只写 Source；
- Target 只做查询、搜索、只读分析。

不要在 Target 上直接写入业务数据，否则会破坏主备一致性。

- CDC 只同步开启后的增量

CDC 不会自动补历史数据。如果集群已有历史数据，需要使用：

- Milvus Backup；
- 恢复到 Target；
- 然后再配置 CDC 增量同步。

-  BulkInsert 暂不支持

当前 CDC 暂不支持同步 BulkInsert 操作。开启 CDC 后，不建议继续使用 BulkInsert。

-  CDC 当前仅支持单副本

Source 集群中：

```
components:
  cdc:
    replicas: 1
```

不要配置多个 CDC 副本。

- 生产环境建议重点监控：

- CDC Pod 是否存活；
- CDC 日志是否持续报错；
- Source 与 Target 的数据延迟；
- Target 存储容量；
- Target QueryNode 资源；
- Source 到 Target 的网络延迟和带宽。



